In [4]:
import json
import plotly.graph_objects as go
file_path = "/home/glaswigian/DL-Estimator/output.json"
# Load the JSON data
with open(file_path, "r") as f:
    data = json.load(f)

# Navigate to the nested SGD data (adjust keys if necessary)
sgd_data = data["VGG16"]["SGD"]

# Initialize lists to store batch numbers and corresponding values
batches = []
torch_values = []
hf_values = []

# Iterate over each batch key in the SGD data
for batch_str, batch_info in sgd_data.items():
    try:
        batch = int(batch_str)
    except ValueError:
        continue  # Skip keys that cannot be converted to int
    # Extract seg_memory_overhead_pre_byte values for torch and huggingface
    torch_val = batch_info.get("torch", {}).get("estimated", {}) \
                         .get("memory", {}).get("segment")
    hf_val = batch_info.get("huggingface", {}).get("estimated", {}) \
                      .get("memory", {}).get("tensor")

    # Only include the batch if at least one of the values exists
    if torch_val is not None or hf_val is not None:
        batches.append(batch)
        torch_values.append(torch_val)
        hf_values.append(hf_val)

# Sort the data by batch number
sorted_data = sorted(zip(batches, torch_values, hf_values), key=lambda x: x[0])
batches, torch_values, hf_values = zip(*sorted_data)

# Create Plotly traces for each line
trace_torch = go.Scatter(
    x=batches,
    y=torch_values,
    mode='lines+markers',
    name='torch'
)
trace_hf = go.Scatter(
    x=batches,
    y=hf_values,
    mode='lines+markers',
    name='huggingface'
)

# Create the figure and update layout
fig = go.Figure(data=[trace_torch, trace_hf])
fig.update_layout(
    title="Segment Memory vs Batch Number",
    xaxis_title="Batch Number",
    yaxis_title="Segment Memory"
)

# Display the figure
fig.show()
